05 Ground Truth Audit

Goal: combine the four ground-truth batches, add complexity/ambiguity/should-move labels, and create review queues.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.features import place_complexity, pin_ambiguity, should_move_rule
from src.metrics import task_aware_report, segmented_task_report

PROCESSED = PROJECT_ROOT / "data" / "processed"

PROJECT_ROOT, PROCESSED


(WindowsPath('c:/Users/aaron/Documents/Pin-To-Place'),
 WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed'))

In [2]:
files = sorted((PROCESSED / "ground_truth").glob("ground_truth_*.csv"))

files

[WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/ground_truth_0_999.csv'),
 WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/ground_truth_1000_1999.csv'),
 WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/ground_truth_2000_2999.csv'),
 WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/ground_truth_3000_3424.csv'),
 WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/ground_truth_combined.csv')]

In [3]:
df = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)

df["place_complexity"] = df.apply(place_complexity, axis=1)
df["pin_ambiguity"] = df.apply(pin_ambiguity, axis=1)
df["should_move"] = df.apply(should_move_rule, axis=1)

df.to_csv(PROCESSED / "ground_truth_combined.csv", index=False)

df.shape

(6850, 20)

In [4]:
task_aware_report(df)

{'count': 6850,
 'mean_m': np.float64(6.4),
 'median_m': np.float64(0.0),
 'p90_m': np.float64(37.55),
 'p95_m': np.float64(40.28),
 'max_m': np.float64(88.56),
 'pct_exact_no_move': np.float64(79.6),
 'pct_over_10m': np.float64(19.2),
 'pct_over_25m': np.float64(14.0),
 'pct_over_50m': np.float64(0.1)}

In [5]:
segmented_task_report(df, "tier_label")


,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
2,418,15.88,8.47,39.72,41.17,78.73,48.3,44.0,37.3,0.5,open_space
3,4614,7.87,0.00,38.46,40.85,47.39,74.9,24.1,17.0,0.0,standard_commercial
0,326,2.69,0.00,0.00,34.75,88.56,93.3,6.7,5.5,0.6,multi_tenant
1,1492,0.00,0.00,0.00,0.00,0.00,100.0,0.0,0.0,0.0,no_building


In [6]:
segmented_task_report(df, "place_complexity")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
0,606,11.67,0.0,39.06,41.02,46.88,61.7,34.3,26.7,0.0,complex
2,6000,5.86,0.0,37.27,40.20,88.56,81.4,17.7,12.7,0.1,simple
1,244,6.61,0.0,38.27,39.84,43.23,80.3,19.7,14.8,0.0,multi_tenant


In [7]:
segmented_task_report(df, "pin_ambiguity")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
1,4128,7.96,0.0,38.52,40.90,88.56,75.0,23.8,17.4,0.1,low
2,662,5.87,0.0,37.02,39.77,47.39,80.1,19.0,11.8,0.0,medium
0,2060,3.43,0.0,10.12,36.85,46.88,88.7,10.1,7.9,0.0,high


In [8]:
review_cols = [
    "id",
    "name",
    "category_primary",
    "region",
    "tier_label",
    "place_complexity",
    "pin_ambiguity",
    "gt_confidence",
    "offset_haversine_m",
    "should_move",
    "gt_reasoning",
]

high_offset = df[df["offset_haversine_m"] >= 30].sort_values(
    "offset_haversine_m",
    ascending=False,
)

high_offset[review_cols].head(75)

,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
4839,08f2ad3c69044aca03c94926717c3b4a,Redbox,rental_kiosks,NC,multi_tenant,simple,low,0.9,88.558515,True,The pin needs to be placed at the specific sto...
1414,08f2ad3c69044aca03c94926717c3b4a,Redbox,rental_kiosks,NC,multi_tenant,simple,low,0.9,88.558515,True,The pin needs to be placed at the specific sto...
3387,08f2ad20087595a6033fe3dd7fb34603,Prestige Funeral Home,funeral_services_and_cemeteries,SC,open_space,simple,low,0.8,78.728265,True,"The pin is placed at the main access point, wh..."
6812,08f2ad20087595a6033fe3dd7fb34603,Prestige Funeral Home,funeral_services_and_cemeteries,SC,open_space,simple,low,0.8,78.728265,True,"The pin is placed at the main access point, wh..."
2511,08f441e71062870d03e54aec9f981198,Best Western,hotel,FL,standard_commercial,simple,medium,0.9,47.393423,True,The pin is placed at the main entrance of the ...
...,...,...,...,...,...,...,...,...,...,...,...
1040,08f4416aa4b2b9400340128c0532c8bc,Paradise Seafood Coffee Shop,food_truck,FL,open_space,simple,low,0.8,43.397155,True,The pin is placed at the main access point nea...
4465,08f4416aa4b2b9400340128c0532c8bc,Paradise Seafood Coffee Shop,food_truck,FL,open_space,simple,low,0.8,43.397155,True,The pin is placed at the main access point nea...
2516,08f441aaa0b3176c03dc377822857192,White Duck Espresso,coffee_shop,FL,standard_commercial,simple,low,0.9,43.345123,True,The pin is placed at the main customer entranc...
5941,08f441aaa0b3176c03dc377822857192,White Duck Espresso,coffee_shop,FL,standard_commercial,simple,low,0.9,43.345123,True,The pin is placed at the main customer entranc...


In [9]:
low_confidence = df[df["gt_confidence"] < 0.6].sort_values(
    ["tier_label", "gt_confidence"],
    ascending=[True, True],
)

low_confidence[review_cols].head(75)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
434,08f2a10681921a2403844334d678de93,KinderZone,day_care_preschool,NJ,multi_tenant,simple,low,0.0,0.0,False,"The current pin is at the center of the image,..."
676,08f2a1225e49c46e0373a4acca5798d4,Childtime of Doylestown,child_care_and_day_care,PA,multi_tenant,simple,low,0.0,0.0,False,"The current pin is at the center of the image,..."
1516,08f44a194dc5a22d03ff93914b1f2cde,Rocky'S Ranch,shopping,FL,multi_tenant,simple,low,0.0,0.0,False,"The current pin is at the center of the image,..."
2622,08f2aa8c2c91654c0307f9ec74d32578,Nicole Matthews Daycare,child_care_and_day_care,MD,multi_tenant,simple,low,0.0,0.0,False,The current pin is at the center of the image ...
3859,08f2a10681921a2403844334d678de93,KinderZone,day_care_preschool,NJ,multi_tenant,simple,low,0.0,0.0,False,"The current pin is at the center of the image,..."
...,...,...,...,...,...,...,...,...,...,...,...
288,08f266886d2026750381a205ba623a9b,Louisville Presbyterian Theological Seminary,professional_services,KY,no_building,simple,high,0.3,0.0,False,The address appears to be a residential area w...
298,08f2ab3850aca311032a958e700a41e6,Chuck Brown II Bail Bonds,bail_bonds_service,OH,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible a...
299,08f489c03369686d039bc697982bc8d3,Brass,event_planning,TX,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible; ...
311,08f29a198ec60c0b03eac222b62b6ef3,Lovet Agency,web_designer,CA,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible a...


In [10]:
multi_tenant = df[df["tier_label"] == "multi_tenant"].sort_values(
    "offset_haversine_m",
    ascending=False,
)

multi_tenant[review_cols].head(75)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
1414,08f2ad3c69044aca03c94926717c3b4a,Redbox,rental_kiosks,NC,multi_tenant,simple,low,0.9,88.558515,True,The pin needs to be placed at the specific sto...
4839,08f2ad3c69044aca03c94926717c3b4a,Redbox,rental_kiosks,NC,multi_tenant,simple,low,0.9,88.558515,True,The pin needs to be placed at the specific sto...
6157,08f441ad446d54200323a1a9871a5d0d,Pinch A Penny Pool Patio Spa,hot_tubs_and_pools,FL,multi_tenant,simple,low,0.9,43.521618,True,The specific unit for 'Pinch A Penny Pool Pati...
2732,08f441ad446d54200323a1a9871a5d0d,Pinch A Penny Pool Patio Spa,hot_tubs_and_pools,FL,multi_tenant,simple,low,0.9,43.521618,True,The specific unit for 'Pinch A Penny Pool Pati...
5796,08f44c0a32819ca303fba357b8406a3c,Redbox,rental_kiosks,GA,multi_tenant,simple,low,0.9,41.318362,True,The pin should be placed at the specific unit'...
...,...,...,...,...,...,...,...,...,...,...,...
1057,08f44dab2161987603d9fd37536e03c4,Childcare Network,day_care_preschool,NC,multi_tenant,simple,low,1.0,0.000000,False,The current pin is already at the correct unit...
1066,08f2664020704ad903d3dc57002e1491,Busy Bees Childcare,day_care_preschool,IN,multi_tenant,simple,low,1.0,0.000000,False,The current pin is already at the correct unit...
1082,08f44d872c69235303fa60eacbcbbd72,Eaves Farm Supply,shopping,NC,multi_tenant,simple,low,1.0,0.000000,False,The current pin is already at the correct unit...
1114,08f279e4a851308b030373f134af2819,Big Timber Daycare,day_care_preschool,MT,multi_tenant,simple,low,1.0,0.000000,False,The current pin is already at the correct unit...


In [11]:
should_move = df[df["should_move"]].sort_values(
    "offset_haversine_m",
    ascending=False,
)

should_move[review_cols].head(75)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
4839,08f2ad3c69044aca03c94926717c3b4a,Redbox,rental_kiosks,NC,multi_tenant,simple,low,0.9,88.558515,True,The pin needs to be placed at the specific sto...
1414,08f2ad3c69044aca03c94926717c3b4a,Redbox,rental_kiosks,NC,multi_tenant,simple,low,0.9,88.558515,True,The pin needs to be placed at the specific sto...
3387,08f2ad20087595a6033fe3dd7fb34603,Prestige Funeral Home,funeral_services_and_cemeteries,SC,open_space,simple,low,0.8,78.728265,True,"The pin is placed at the main access point, wh..."
6812,08f2ad20087595a6033fe3dd7fb34603,Prestige Funeral Home,funeral_services_and_cemeteries,SC,open_space,simple,low,0.8,78.728265,True,"The pin is placed at the main access point, wh..."
5936,08f441e71062870d03e54aec9f981198,Best Western,hotel,FL,standard_commercial,simple,medium,0.9,47.393423,True,The pin is placed at the main entrance of the ...
...,...,...,...,...,...,...,...,...,...,...,...
1040,08f4416aa4b2b9400340128c0532c8bc,Paradise Seafood Coffee Shop,food_truck,FL,open_space,simple,low,0.8,43.397155,True,The pin is placed at the main access point nea...
4465,08f4416aa4b2b9400340128c0532c8bc,Paradise Seafood Coffee Shop,food_truck,FL,open_space,simple,low,0.8,43.397155,True,The pin is placed at the main access point nea...
5941,08f441aaa0b3176c03dc377822857192,White Duck Espresso,coffee_shop,FL,standard_commercial,simple,low,0.9,43.345123,True,The pin is placed at the main customer entranc...
2516,08f441aaa0b3176c03dc377822857192,White Duck Espresso,coffee_shop,FL,standard_commercial,simple,low,0.9,43.345123,True,The pin is placed at the main customer entranc...


In [12]:
zero_offset_sample = (
    df[df["offset_haversine_m"] == 0]
    .sample(n=min(150, (df["offset_haversine_m"] == 0).sum()), random_state=42)
)

zero_offset_sample[review_cols].head(75)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,should_move,gt_reasoning
6476,08f2a9194b12d6e3035f949bc6203527,Crockett's Run,event_planning,OH,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible; ...
5904,08f268cc350735ae03e0a00113a3c1b0,prodriguez,software_development,CO,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible a...
5968,08f276aceb472b8e03839c14f416918a,Sojourn Lakeside Resort,hotel,NaN,standard_commercial,complex,high,1.0,0.0,False,The current pin is already at the main custome...
3425,08f266694b8dd34103db5d8f268c8bfe,SNT Biotech Lab,pharmacy,IL,standard_commercial,simple,low,1.0,0.0,False,The current pin is already at the main custome...
2462,08f2ab6426921ccb03bb33ea6889a5b1,Kings Ventures Custom Processing,butcher_shop,MI,standard_commercial,simple,low,1.0,0.0,False,The current pin is already at the main custome...
...,...,...,...,...,...,...,...,...,...,...,...
2824,08f2a996334b519103a8b07eca0662da,"FlexTech Solutions, LLC",professional_services,TN,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible; ...
6573,08f2ad2968d54c6203888cf63a084922,Smithfield Packing,professional_services,NC,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible; ...
3392,08f446ca98b4d4850394bdfc2defb457,Shred Nations,shredding_services,TX,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible a...
6266,08f2830b80ba095c03a25283513235ea,Scot Candell & Associates,lawyer,CA,no_building,simple,high,0.3,0.0,False,No dedicated commercial structure is visible a...


In [13]:
outputs = {
    "review_high_offset.csv": high_offset,
    "review_low_confidence.csv": low_confidence,
    "review_multi_tenant.csv": multi_tenant,
    "review_should_move.csv": should_move,
    "review_zero_offset_sample.csv": zero_offset_sample,
}

for filename, frame in outputs.items():
    frame.to_csv(PROCESSED / filename, index=False)

list(outputs.keys())


['review_high_offset.csv',
 'review_low_confidence.csv',
 'review_multi_tenant.csv',
 'review_should_move.csv',
 'review_zero_offset_sample.csv']

In [14]:
summary_lines = []

summary_lines.append("Ground Truth Audit Summary")
summary_lines.append("")
summary_lines.append("Overall")
summary_lines.append(str(task_aware_report(df)))
summary_lines.append("")
summary_lines.append("By tier")
summary_lines.append(segmented_task_report(df, "tier_label").to_string(index=False))
summary_lines.append("")
summary_lines.append("By place complexity")
summary_lines.append(segmented_task_report(df, "place_complexity").to_string(index=False))
summary_lines.append("")
summary_lines.append("By ambiguity")
summary_lines.append(segmented_task_report(df, "pin_ambiguity").to_string(index=False))
summary_lines.append("")
summary_lines.append(f"High-offset review rows: {len(high_offset)}")
summary_lines.append(f"Low-confidence review rows: {len(low_confidence)}")
summary_lines.append(f"Multi-tenant review rows: {len(multi_tenant)}")
summary_lines.append(f"Should-move rows: {len(should_move)}")
summary_lines.append(f"Zero-offset sample rows: {len(zero_offset_sample)}")

summary_path = PROCESSED / "audit" / "ground_truth_audit_summary.txt"
summary_path.write_text("\n".join(summary_lines))

summary_path


WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/ground_truth_audit_summary.txt')